Часть 2. Марковские цепи. Матрица переходов для ДНК 1-го порядка

In [2]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 39.9 MB/s eta 0:00:00


In [8]:
import numpy as np
from Bio import SeqIO

# Загрузка последовательности, сгенерируем демонстрационную последовательность
def load_sequence():
    try:
        record = SeqIO.read("sequence.fasta", "fasta")
        return str(record.seq)
    except FileNotFoundError:

        rng = np.random.default_rng(123)

        nucs = ['A','C','G','T']
        seq = [rng.choice(nucs)]

        trans_mat = np.array([[0.3, 0.2, 0.2, 0.3],
                              [0.25,0.3,0.25,0.2],
                              [0.2,0.3,0.3,0.2],
                              [0.3,0.2,0.2,0.3]])
        idx = {'A':0,'C':1,'G':2,'T':3}
        for _ in range(9999):
            prev = seq[-1]
            seq.append(rng.choice(nucs, p=trans_mat[idx[prev]]))
        return ''.join(seq)

seq = load_sequence()
print(f"Длина последовательности: {len(seq)}")

# Частоты 16 динуклеотидов
nucleotides = ['A','C','G','T']
idx_map = {n:i for i,n in enumerate(nucleotides)}
dim = 4
dinuc_counts = np.zeros((dim, dim), dtype=int)

for i in range(len(seq)-1):
    first = idx_map[seq[i]]
    second = idx_map[seq[i+1]]
    dinuc_counts[first, second] += 1

print("Матрица частот динуклеотидов:\n", dinuc_counts)

#  Матрица переходов P_ij = P(j|i)
P = dinuc_counts / dinuc_counts.sum(axis=1, keepdims=True)
print("\nМатрица переходов P (4x4):\n", P)

row_sums = P.sum(axis=1)
print("Суммы строк P:", row_sums)

# -Стационарное распределение π
eigenvalues, eigenvectors = np.linalg.eig(P.T)

idx = np.argmin(np.abs(eigenvalues - 1.0))
pi = np.real(eigenvectors[:, idx])
pi = pi / pi.sum()
print("\nСтационарное распределение π:", pi)
print(f"  A: {pi[0]:.4f}, C: {pi[1]:.4f}, G: {pi[2]:.4f}, T: {pi[3]:.4f}")

obs_counts = np.array([seq.count(n) for n in nucleotides], dtype=float)
obs_freq = obs_counts / len(seq)
print("\nНаблюдаемые частоты нуклеотидов:", dict(zip(nucleotides, obs_freq)))
print("Стационарные частоты π:", dict(zip(nucleotides, pi)))


Длина последовательности: 10000
Матрица частот динуклеотидов:
 [[800 534 535 783]
 [640 791 628 469]
 [488 710 777 471]
 [723 494 506 650]]

Матрица переходов P (4x4):
 [[0.30165913 0.20135747 0.20173454 0.29524887]
 [0.25316456 0.31289557 0.24841772 0.18552215]
 [0.1995094  0.29026983 0.31766149 0.19255928]
 [0.30467762 0.20817531 0.2132322  0.27391488]]
Суммы строк P: [1. 1. 1. 1.]

Стационарное распределение π: [0.2651204  0.25293833 0.24463027 0.237311  ]
  A: 0.2651, C: 0.2529, G: 0.2446, T: 0.2373

Наблюдаемые частоты нуклеотидов: {'A': np.float64(0.2652), 'C': np.float64(0.2529), 'G': np.float64(0.2446), 'T': np.float64(0.2373)}
Стационарные частоты π: {'A': np.float64(0.265120399121101), 'C': np.float64(0.25293833100026564), 'G': np.float64(0.24463026638971352), 'T': np.float64(0.23731100348891987)}
